# Notebook 02: Contextual Embeddings

## Why This Matters
Static embeddings give every word one fixed vector. "Bank" in a financial context and "bank" in a geographic context are indistinguishable. This matters: sentiment analysis, named entity recognition, question answering, and nearly every real NLP task requires knowing what a word means *in context*, not in the abstract.

ELMo (2018) established the idea. BERT (2018) made it the standard. Sentence-BERT (2019) made it practical for retrieval at scale. These three models represent the shift that made the current era of NLP possible — and sentence-transformers are the tool you should reach for today when you need embeddings.

## Structure
1. Why context matters — the polysemy problem (narrative)
2. ELMo — bidirectional LSTMs and why the idea worked (narrative)
3. BERT — masked language modeling and deep bidirectionality (narrative)
4. Sentence-BERT — pooling BERT for sentence-level similarity (code)
5. Evaluating embedding quality with STS benchmarks (code)
6. Practical retrieval: encode, index, search (code)
7. Bridge to contrastive learning

## What You'll Learn
- Why ELMo's key insight (context-dependent vectors) was the decisive break from static embeddings
- How BERT's masked LM objective produces deeply bidirectional representations
- How sentence-transformers adapt BERT for efficient similarity search with mean pooling + contrastive fine-tuning
- How to evaluate embedding quality on semantic textual similarity benchmarks
- How to build a practical encode-index-search retrieval pipeline with sentence-transformers


## Background

### The polysemy problem

A static embedding for "bank" is forced to average all its senses into one point. The model that produced it saw "river bank", "bank account", and "blood bank" — and blended them. No downstream task can recover the distinction.

This isn't just a word-sense problem. Even unambiguous words carry contextual nuance. "Heavy" in "heavy traffic" and "heavy meal" have different pragmatic meanings. "Run" in a biological paper means something different than in a sports article. Static embeddings encode none of this.

### ELMo: context via bidirectional LSTMs

ELMo (Embeddings from Language Models, Peters et al., 2018) was the first widely-deployed fix. It trains a deep bidirectional LSTM as a language model — predicting the next word from the left and the previous word from the right simultaneously. The hidden states at each layer become the word representation.

The key property: the LSTM must read the full context on both sides before producing a representation. "Bank" in "I went to the **bank** to deposit money" gets different hidden states than in "we sat on the **bank** of the river" — because the surrounding words drove the LSTM into different states.

ELMo representations are **task-agnostic**: freeze the LSTM, extract the hidden states for any sequence, and use them as features for any downstream model. It improved the state of the art on 6 major NLP benchmarks simultaneously at release.

**Why it's largely historical:** ELMo's LSTM is slower and less capable than a transformer. BERT made it obsolete within a year, and the field has never looked back at bidirectional LSTMs.

### BERT: masked language modeling and the transformer encoder

BERT (Devlin et al., 2018) replaced ELMo's bidirectional LSTM with a transformer encoder and a new pretraining objective: **masked language modeling (MLM)**.

Instead of predicting the next token left-to-right (as GPT does), BERT randomly masks 15% of tokens and trains the model to predict them using context from *both directions simultaneously*:

```
Input:  "The [MASK] sat on the mat"
Target: "cat"
```

Because every token attends to every other token via self-attention, the representation for each masked token is computed using full bidirectional context from the start — not recurrently like ELMo, but in parallel via attention.

The result: token representations that encode rich contextual information. "Bank" in a financial sentence and "bank" in a geographic sentence produce different hidden states at every layer — not just the output, but all 12 layers.

**BERT's limitation for retrieval:** BERT produces one vector per *token*, not one vector per *sentence*. To use it for similarity search, you need to pool those token vectors into a fixed-size sentence representation. The naive approach — taking the `[CLS]` token — works poorly. Mean pooling is better. Contrastive fine-tuning on sentence pairs is what actually works.

### Sentence-BERT: contrastive fine-tuning for retrieval

Sentence-BERT (Reimers & Gurevych, 2019) fine-tunes BERT with a **siamese network** structure on sentence pairs labeled for semantic similarity (NLI and STS datasets):

```
encode(sentence_A) → mean-pool → vector_A
encode(sentence_B) → mean-pool → vector_B
cosine_similarity(vector_A, vector_B) → similarity score
```

The contrastive objective pulls semantically similar sentences together and pushes dissimilar ones apart in embedding space. The result: sentence vectors where cosine similarity is a meaningful proxy for semantic similarity — useful for retrieval, clustering, and semantic search.

The `sentence-transformers` library (which wraps this) is the right tool for any embedding task where you need sentence-level vectors today.


In [ ]:
# Self-contained install — run once
# %pip install -q sentence-transformers torch numpy scikit-learn matplotlib

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer, util

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Device: {device}")

# all-MiniLM-L6-v2: 22M params, 384-dim, excellent speed/quality tradeoff
# Downloads ~90MB on first run
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
print(f"Model: all-MiniLM-L6-v2")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

## 1. Context-Dependent Vectors

The first thing to verify: the same word in different contexts produces different vectors. This is what BERT enables and static embeddings cannot do.

In [ ]:
# "bank" in two clearly different senses
financial_sentences = [
    "I went to the bank to deposit my paycheck.",
    "The bank approved my mortgage application.",
    "She invested her savings in the bank's high-yield account.",
    "The central bank raised interest rates to combat inflation.",
]

geographic_sentences = [
    "We sat on the river bank and watched the sunset.",
    "The fisherman cast his line from the grassy bank.",
    "Erosion is wearing away the steep bank of the stream.",
    "They camped on the bank of the Colorado River.",
]

all_sentences = financial_sentences + geographic_sentences
labels = ["financial"] * 4 + ["geographic"] * 4

embeddings = model.encode(all_sentences, convert_to_tensor=True)

print("Cosine similarities — 'bank' contexts:")
print()
print("Financial vs Financial (should be HIGH):")
for i in range(len(financial_sentences)):
    for j in range(i+1, len(financial_sentences)):
        sim = util.cos_sim(embeddings[i], embeddings[j]).item()
        print(f"  {sim:.3f}  {financial_sentences[i][:40]!r} ↔ {financial_sentences[j][:40]!r}")

print()
print("Financial vs Geographic (should be LOW):")
for i in range(len(financial_sentences)):
    sim = util.cos_sim(embeddings[i], embeddings[4]).item()
    print(f"  {sim:.3f}  {financial_sentences[i][:40]!r} ↔ {geographic_sentences[0][:40]!r}")

## 2. Semantic Textual Similarity

STS (Semantic Textual Similarity) is the standard benchmark for embedding quality. Human annotators score sentence pairs 0–5 on semantic similarity; we measure Spearman correlation between those scores and our cosine similarities.

In [ ]:
# A small STS-style evaluation set — human similarity scores 0-5
sts_pairs = [
    ("A man is playing a guitar.", "A person is playing a musical instrument.", 4.5),
    ("The cat sat on the mat.", "A dog is sleeping on the floor.", 1.0),
    ("She is running in the park.", "The woman jogs through the garden.", 4.0),
    ("The bank raised interest rates.", "We fished along the river bank.", 0.5),
    ("Two men are playing chess.", "Two people are playing a board game.", 3.5),
    ("The economy is growing rapidly.", "GDP increased significantly this quarter.", 4.2),
    ("He ate a sandwich for lunch.", "She had pizza for dinner.", 2.0),
    ("The flight was delayed by two hours.", "The plane arrived late.", 3.8),
    ("A child is drawing a picture.", "A kid is painting.", 3.9),
    ("It is raining outside.", "The weather is sunny and warm.", 0.3),
]

sentences_a = [p[0] for p in sts_pairs]
sentences_b = [p[1] for p in sts_pairs]
human_scores = np.array([p[2] for p in sts_pairs])

emb_a = model.encode(sentences_a, convert_to_tensor=True)
emb_b = model.encode(sentences_b, convert_to_tensor=True)
cosine_scores = util.cos_sim(emb_a, emb_b).diagonal().cpu().numpy()

from scipy.stats import spearmanr
corr, pval = spearmanr(human_scores, cosine_scores)

print(f"Spearman correlation with human judgments: {corr:.3f}  (p={pval:.4f})")
print(f"(Full STS-B benchmark: all-MiniLM-L6-v2 scores ~0.88)")
print()
print(f"{'Human':>6}  {'Model':>6}  Sentence pair")
print("-" * 75)
for (s1, s2, human), model_score in zip(sts_pairs, cosine_scores):
    print(f"{human:>6.1f}  {model_score:>6.3f}  {s1[:30]!r} ↔ {s2[:30]!r}")

## 3. Practical Retrieval: Encode → Index → Search

The main production use case for sentence-transformers is semantic search — encode a corpus once, then find the most similar documents to any query at inference time.

In [ ]:
# A small knowledge base to search over
knowledge_base = [
    "Flash attention reduces memory complexity from O(N²) to O(N) by tiling.",
    "The KV cache stores key and value tensors to avoid recomputing attention.",
    "Speculative decoding uses a draft model to propose tokens, then verifies in parallel.",
    "Quantization reduces model precision from FP32 to INT8 or INT4 to save memory.",
    "Continuous batching allows new requests to join an in-flight batch mid-generation.",
    "PagedAttention manages KV cache memory using virtual memory paging.",
    "Temperature scaling controls the sharpness of the output probability distribution.",
    "Beam search maintains multiple candidate sequences and picks the highest-probability one.",
    "Rotary position embeddings (RoPE) encode position via rotation in the complex plane.",
    "Mixture of experts routes each token to a subset of expert FFN layers.",
    "Gradient checkpointing trades compute for memory during backpropagation.",
    "LoRA fine-tunes only low-rank decomposition matrices, not the full weight matrices.",
    "RLHF aligns model outputs with human preferences using a learned reward model.",
    "Context length extension via YaRN rescales RoPE frequencies for longer sequences.",
    "vLLM uses PagedAttention and continuous batching to maximize GPU inference throughput.",
]

# Encode corpus once — in production this is done offline and cached
corpus_embeddings = model.encode(knowledge_base, convert_to_tensor=True, show_progress_bar=False)
print(f"Encoded {len(knowledge_base)} documents into {corpus_embeddings.shape} tensor")

In [ ]:
def semantic_search(query: str, corpus_embeddings, knowledge_base, top_k=3):
    query_embedding = model.encode(query, convert_to_tensor=True)
    hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=top_k)[0]
    return [(knowledge_base[h["corpus_id"]], h["score"]) for h in hits]


queries = [
    "how does attention memory scale with sequence length?",
    "what techniques reduce GPU memory usage during inference?",
    "how do you serve multiple users efficiently?",
    "how does position information get encoded in transformers?",
]

for query in queries:
    print(f"Query: {query!r}")
    results = semantic_search(query, corpus_embeddings, knowledge_base, top_k=2)
    for doc, score in results:
        print(f"  [{score:.3f}] {doc}")
    print()

## 4. Choosing a Model

`sentence-transformers` hosts dozens of pretrained models. The tradeoff is always speed vs quality vs dimension size.

In [ ]:
# Model comparison — encode speed and embedding quality tradeoffs
# These are the commonly recommended options as of 2024

model_options = {
    "all-MiniLM-L6-v2":        {"params": "22M",  "dim": 384,  "sts_b": 0.886, "notes": "Best speed/quality, default choice"},
    "all-mpnet-base-v2":        {"params": "109M", "dim": 768,  "sts_b": 0.921, "notes": "Higher quality, 5x slower"},
    "multi-qa-MiniLM-L6-cos-v1":{"params": "22M",  "dim": 384,  "sts_b": 0.876, "notes": "Optimized for retrieval (asymmetric)"},
    "paraphrase-multilingual-MiniLM-L12-v2": {"params": "118M", "dim": 384, "sts_b": 0.835, "notes": "50+ languages"},
    "text-embedding-3-small (OpenAI)": {"params": "N/A", "dim": 1536, "sts_b": 0.926, "notes": "API only, strong baseline"},
}

print(f"{'Model':<45} {'Params':>8} {'Dim':>6} {'STS-B':>7}  Notes")
print("-" * 95)
for name, info in model_options.items():
    print(f"{name:<45} {info['params']:>8} {info['dim']:>6} {info['sts_b']:>7.3f}  {info['notes']}")

print()
print("Rule of thumb:")
print("  Default:          all-MiniLM-L6-v2")
print("  Need multilingual: paraphrase-multilingual-MiniLM-L12-v2")
print("  Need highest quality: all-mpnet-base-v2 or text-embedding-3-small")
print("  Need asymmetric Q&A retrieval: multi-qa-MiniLM-L6-cos-v1")

## 5. Visualizing the Embedding Space

In [ ]:
# Encode a set of semantically related sentences and project to 2D
groups = {
    "inference optimization": [
        "KV cache stores attention keys and values",
        "Speculative decoding accelerates token generation",
        "Flash attention is memory efficient",
        "Continuous batching improves GPU utilization",
    ],
    "model compression": [
        "Quantization reduces model precision to INT8",
        "LoRA fine-tunes low-rank matrices only",
        "Pruning removes less important weights",
        "Knowledge distillation trains a smaller student model",
    ],
    "alignment": [
        "RLHF aligns models with human preferences",
        "Constitutional AI uses self-critique for alignment",
        "Reward models score response quality",
        "DPO directly optimizes preference pairs",
    ],
}

all_sents, all_labels, all_colors = [], [], []
color_map = {"inference optimization": "royalblue", "model compression": "darkorange", "alignment": "seagreen"}

for group, sents in groups.items():
    all_sents.extend(sents)
    all_labels.extend([group] * len(sents))
    all_colors.extend([color_map[group]] * len(sents))

embs = model.encode(all_sents)
coords = PCA(n_components=2).fit_transform(embs)

fig, ax = plt.subplots(figsize=(11, 7))
for i, (sent, label, color) in enumerate(zip(all_sents, all_labels, all_colors)):
    ax.scatter(*coords[i], c=color, s=80, zorder=2)
    ax.annotate(sent[:40], coords[i], fontsize=7.5, ha="center", va="bottom",
                xytext=(0, 5), textcoords="offset points")

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=g) for g, c in color_map.items()], loc="upper right")
ax.set_title("Sentence Embeddings — PCA Projection (all-MiniLM-L6-v2)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Semantically related sentences cluster together even with different surface forms.")

## 6. Bridge to Contrastive Learning

Sentence-BERT is fine-tuned with a contrastive objective: pull similar sentences together, push dissimilar ones apart. This is the same structure as SimCLR and CLIP — just applied to text pairs instead of image augmentations or image-caption pairs.

The pattern:
```
encode(A) → vector_A
encode(B) → vector_B
loss = pull_together if similar else push_apart
```

SimCLR (Notebook 06) applies this to image augmentation pairs. MoCo (Notebook 07) adds a memory queue to scale it. CLIP (Notebook 09) applies it across modalities — image and caption as positive pairs, everything else as negatives.

Understanding sentence-BERT's training objective makes all of these immediately legible.

**Next:** Notebook 03 covers fine-tuning embeddings for specific domains — when the general pretrained model isn't good enough and you need to adapt it to your data.


## Summary

| Model | Year | Key Idea | Status |
|-------|------|----------|--------|
| ELMo | 2018 | Bidirectional LSTM; context-dependent vectors | Historical — understand the idea, don't deploy |
| BERT | 2018 | Transformer encoder + masked LM pretraining | Foundation — use via sentence-transformers |
| Sentence-BERT | 2019 | Contrastive fine-tuning for sentence similarity | **Active** — the right tool for embedding tasks today |

**Key practical points:**
- `all-MiniLM-L6-v2` is the default starting point for sentence-level embeddings
- Semantic search = encode corpus offline → cosine search at query time
- STS-B Spearman correlation is the standard benchmark for comparing embedding models
- Sentence-transformers contrastive objective is the same structure as SimCLR/CLIP

**Next:** Notebook 03 — Fine-tuning embeddings for domain-specific tasks.
